# obp: LP & MIP across three solver interfaces

Tests the `obp` modeling DSL against three MIP-capable backends:

- **HiGHS** (`highspy`) — MIT licensed, bundled, always available
- **SCIP** (`pyscipopt`) — free academic/permissive license, install via `pip install pyscipopt`
- **Gurobi** (`gurobipy`) — commercial, requires a license; cells auto-skip if unavailable

Each solver is checked on the same LP and MIP problems so results can be compared directly.


## Setup

Locate the repo root and import `obp`.

In [1]:
from pathlib import Path
import sys

def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "CMakeLists.txt").is_file() and (candidate / "python").is_dir():
            return candidate
    raise RuntimeError("Could not find the solvers repository root from the current directory")

ROOT = find_repo_root(Path.cwd())
sys.path.insert(0, str(ROOT / "python"))

import numpy as np
from obp import Problem, solve

print(f"repository: {ROOT}")


repository: /data/dev/solvers


In [2]:
def solver_available(name: str) -> bool:
    try:
        if name == "highs":
            import highspy  # noqa: F401
        elif name == "scip":
            import pyscipopt  # noqa: F401
        elif name == "gurobi":
            import gurobipy  # noqa: F401
        else:
            return False
        return True
    except ImportError:
        return False

SOLVERS = [s for s in ("highs", "scip", "gurobi") if solver_available(s)]
print("Available MIP-capable solvers:", SOLVERS)
if not SOLVERS:
    raise RuntimeError("No MIP-capable solver available — install highspy, pyscipopt, or gurobipy")


Available MIP-capable solvers: ['highs', 'scip', 'gurobi']


## 1. Linear Programming (LP)

```
minimize   3*x0 + x1 + 2*x2
subject to x0 + x1 + x2 == 4
           2*x0 + 0.5*x1 + x2 <= 5
           x0 <= 3
           x0, x1, x2 >= 0
```


In [3]:
def build_lp():
    pb = Problem("blend")
    x = pb.add_variables("x", 3, lb=0)
    pb.set_objective(3 * x[0] + 1 * x[1] + 2 * x[2])
    pb.add_constraint(x[0] + x[1] + x[2] == 4, "sum_eq")
    pb.add_constraint(2 * x[0] + 0.5 * x[1] + x[2] <= 5, "mix_limit")
    pb.add_constraint(x[0] <= 3, "x0_cap")
    return pb


for name in SOLVERS:
    pb = build_lp()
    result = solve(pb, solver=name)
    print(f"[{name:7s}] status={result.status:10s} obj={result.obj_val:.6f} x={np.round(result.x, 4)}")


[highs  ] status=solved     obj=4.000000 x=[-0.  4. -0.]
[scip   ] status=solved     obj=4.000000 x=[0. 4. 0.]
Set parameter Username


Set parameter LicenseID to value 2773151


Academic license - for non-commercial use only - expires 2027-01-30


[gurobi ] status=solved     obj=4.000000 x=[0. 4. 0.]


In [4]:
# All three should agree on the optimum.
results = {}
for name in SOLVERS:
    results[name] = solve(build_lp(), solver=name)

obj_vals = [r.obj_val for r in results.values()]
assert max(obj_vals) - min(obj_vals) < 1e-5, f"LP objective mismatch across solvers: {results}"
print("OK: all solvers agree on the LP optimum:", obj_vals[0])


OK: all solvers agree on the LP optimum: 4.0


## 2. Mixed-Integer Programming (MIP)

```
minimize   -x0 - 2*x1 + b
subject to x0 + x1 + b <= 3
           2*x0 + x1 <= 4
           x0 >= 1
           x0, x1 integer >= 0
           b binary
```

Expected optimum: `x0=1, x1=2, b=0, obj=-5`.


In [5]:
def build_mip():
    pb = Problem("mip_demo")
    x = pb.add_variables("x", 2, vtype="integer", lb=0, ub=10)
    b = pb.add_variables("b", 1, vtype="binary")
    pb.set_objective(-x[0] - 2 * x[1] + b)
    pb.add_constraint(x[0] + x[1] + b <= 3, "sum_limit")
    pb.add_constraint(2 * x[0] + x[1] <= 4, "mix_limit")
    pb.add_constraint(x[0] >= 1, "x0_lower")
    return pb


for name in SOLVERS:
    pb = build_mip()
    result = solve(pb, solver=name)
    print(f"[{name:7s}] status={result.status:10s} obj={result.obj_val:.6f} x={np.round(result.x, 4)}")
    assert result.status == "solved"
    assert abs(result.obj_val - (-5.0)) < 1e-5
    np.testing.assert_allclose(result.x, [1, 2, 0], atol=1e-6)

print("OK: all solvers agree on the MIP optimum: x=[1, 2, 0], obj=-5")


[highs  ] status=solved     obj=-5.000000 x=[ 1.  2. -0.]
[scip   ] status=solved     obj=-5.000000 x=[1. 2. 0.]
[gurobi ] status=solved     obj=-5.000000 x=[1. 2. 0.]
OK: all solvers agree on the MIP optimum: x=[1, 2, 0], obj=-5


## 3. Binary knapsack-style MIP

```
maximize   5*b0 + 4*b1 + 3*b2 + 7*b3
subject to 2*b0 + 3*b1 + 4*b2 + 5*b3 <= 7
           b binary
```


In [6]:
def build_knapsack():
    pb = Problem("knapsack")
    b = pb.add_variables("b", 4, vtype="binary")
    pb.set_objective(5 * b[0] + 4 * b[1] + 3 * b[2] + 7 * b[3], sense="maximize")
    pb.add_constraint(2 * b[0] + 3 * b[1] + 4 * b[2] + 5 * b[3] <= 7, "capacity")
    return pb


knapsack_results = {}
for name in SOLVERS:
    result = solve(build_knapsack(), solver=name)
    knapsack_results[name] = result
    print(f"[{name:7s}] status={result.status:10s} obj={result.obj_val:.6f} x={np.round(result.x, 4)}")

obj_vals = [r.obj_val for r in knapsack_results.values()]
assert max(obj_vals) - min(obj_vals) < 1e-5, f"Knapsack objective mismatch: {knapsack_results}"
print("OK: all solvers agree on the knapsack optimum:", obj_vals[0])


[highs  ] status=solved     obj=12.000000 x=[ 1. -0. -0.  1.]
[scip   ] status=solved     obj=12.000000 x=[1. 0. 0. 1.]
[gurobi ] status=solved     obj=12.000000 x=[1. 0. 0. 1.]
OK: all solvers agree on the knapsack optimum: 12.0


## 4. SOS1 / SOS2 constraints

SOS1: at most one variable nonzero. SOS2: at most two, and they must be adjacent
in the given order. Both are reformulated internally as a MIP (binary indicators)
and solved via whichever of HiGHS/SCIP/Gurobi is selected.


In [7]:
def build_sos1():
    pb = Problem("sos1_demo")
    x = pb.add_variables("x", 3, lb=0, ub=10)
    pb.set_objective(2 * x[0] + 5 * x[1] + 3 * x[2], sense="maximize")
    pb.add_constraint(x[0] + x[1] + x[2] <= 10, "budget")
    pb.add_sos_constraint(x, type=1)
    return pb


for name in SOLVERS:
    result = solve(build_sos1(), solver=name)
    nonzero = int((result.x > 1e-6).sum())
    print(f"[{name:7s}] status={result.status:10s} obj={result.obj_val:.6f} x={np.round(result.x, 4)} nonzero={nonzero}")
    assert result.status == "solved"
    assert abs(result.obj_val - 50.0) < 1e-5
    assert nonzero <= 1

print("OK: SOS1 picks the single best variable (x1=10, obj=50) on every solver")


[highs  ] status=solved     obj=50.000000 x=[ 0. 10.  0.] nonzero=1
[scip   ] status=solved     obj=50.000000 x=[ 0. 10.  0.] nonzero=1
[gurobi ] status=solved     obj=50.000000 x=[ 0. 10.  0.] nonzero=1
OK: SOS1 picks the single best variable (x1=10, obj=50) on every solver


In [8]:
def build_sos2():
    pb = Problem("sos2_demo")
    y = pb.add_variables("y", 4, lb=0, ub=10)
    pb.set_objective(y[0] + y[1] + y[2] + y[3], sense="maximize")
    pb.add_constraint(y[0] + y[1] + y[2] + y[3] <= 10, "budget")
    pb.add_sos_constraint(y, type=2)
    return pb


for name in SOLVERS:
    result = solve(build_sos2(), solver=name)
    nonzero_idx = [i for i, v in enumerate(result.x) if v > 1e-6]
    print(f"[{name:7s}] status={result.status:10s} obj={result.obj_val:.6f} x={np.round(result.x, 4)} nonzero_idx={nonzero_idx}")
    assert result.status == "solved"
    assert abs(result.obj_val - 10.0) < 1e-5
    assert len(nonzero_idx) <= 2
    if len(nonzero_idx) == 2:
        assert abs(nonzero_idx[0] - nonzero_idx[1]) == 1

print("OK: SOS2 respects the cardinality/adjacency rule on every solver")


[highs  ] status=solved     obj=10.000000 x=[10.  0.  0.  0.] nonzero_idx=[0]
[scip   ] status=solved     obj=10.000000 x=[ 0.  0.  0. 10.] nonzero_idx=[3]
[gurobi ] status=solved     obj=10.000000 x=[ 0. 10.  0.  0.] nonzero_idx=[1]
OK: SOS2 respects the cardinality/adjacency rule on every solver


## 5. Solver-specific options

Each backend accepts a small set of common options (time limits, gap tolerances).
Names differ slightly per backend; this cell shows the mapping actually accepted
by each `solve()` call today.


In [9]:
for name in SOLVERS:
    pb = build_mip()
    kwargs = {"time_limit": 5.0}
    result = solve(pb, solver=name, **kwargs)
    print(f"[{name:7s}] time_limit=5.0 -> status={result.status}, obj={result.obj_val:.6f}")


[highs  ] time_limit=5.0 -> status=solved, obj=-5.000000
[scip   ] time_limit=5.0 -> status=solved, obj=-5.000000
[gurobi ] time_limit=5.0 -> status=solved, obj=-5.000000


## Summary

In [10]:
print("Solvers tested:", SOLVERS)
print("All LP, MIP, knapsack, SOS1 and SOS2 checks passed for every available solver.")


Solvers tested: ['highs', 'scip', 'gurobi']
All LP, MIP, knapsack, SOS1 and SOS2 checks passed for every available solver.
